In [ ]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)

RAW_DIR = '../data/raw'

files = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

dfs = {}
for name, filename in files.items():
    path = os.path.join(RAW_DIR, filename)
    dfs[name] = pd.read_csv(path)
    print(f'{name:22s} loaded  ->  shape: {dfs[name].shape}')

In [ ]:
for name, df in dfs.items():
    print(f'\n===== {name} =====')
    print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
    print(df.dtypes)

In [ ]:
for name, df in dfs.items():
    null_counts = df.isnull().sum()
    null_pct = (df.isnull().mean() * 100).round(2)
    nulls = pd.DataFrame({'missing_count': null_counts, 'missing_pct': null_pct})
    nulls = nulls[nulls['missing_count'] > 0]
    if not nulls.empty:
        print(f'\n===== {name}: columns with missing values =====')
        print(nulls)
    else:
        print(f'\n===== {name}: no missing values =====')

In [ ]:
for name, df in dfs.items():
    dupes = df.duplicated().sum()
    print(f'{name:22s} duplicate rows: {dupes}')

In [ ]:
cust = dfs['customers']
print('Unique customer_id:', cust['customer_id'].nunique())
print('Unique customer_unique_id:', cust['customer_unique_id'].nunique())
print('Total rows:', len(cust))

In [ ]:
print(dfs['orders']['order_status'].value_counts())

In [ ]:
orders = dfs['orders'].copy()
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

impossible = orders[orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']]
print('Orders delivered before purchase date:', len(impossible))

In [ ]:
items = dfs['order_items']
print('Rows with price <= 0:', (items['price'] <= 0).sum())
print('Rows with freight_value < 0:', (items['freight_value'] < 0).sum())
print(items[['price', 'freight_value']].describe())

In [ ]:
orphan_products = ~items['product_id'].isin(dfs['products']['product_id'])
orphan_sellers = ~items['seller_id'].isin(dfs['sellers']['seller_id'])
print('order_items rows with product_id not in products table:', orphan_products.sum())
print('order_items rows with seller_id not in sellers table:', orphan_sellers.sum())

In [ ]:
for name, df in dfs.items():
    print(f'\n===== {name} sample =====')
    display(df.head(3))